# Exercícios — Clustering e PCA

Soluções em `# @title`.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

## Exercício 1 — Cotovelo e silhueta

In [ ]:
# @title Solução
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

w = load_wine()
X = StandardScaler().fit_transform(w.data)
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMENTE).fit(X)
    print("k", k, "| inercia", round(km.inertia_, 1), "| silhueta", round(silhouette_score(X, km.labels_), 3))

## Exercício 2 — Padronizar antes de agrupar

In [ ]:
# @title Solução
from sklearn.metrics import adjusted_rand_score

g_cru = KMeans(n_clusters=3, n_init=10, random_state=SEMENTE).fit_predict(w.data)
g_pad = KMeans(n_clusters=3, n_init=10, random_state=SEMENTE).fit_predict(X)
print("ARI sem padronizar:", round(adjusted_rand_score(w.target, g_cru), 3))
print("ARI com padronizar:", round(adjusted_rand_score(w.target, g_pad), 3))
print("variaveis de escala grande (ex.: proline) dominam a distancia sem padronizacao.")

## Exercício 3 — Quantas componentes reter?

In [ ]:
# @title Solução
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA

bc = load_breast_cancer()
Xbc = StandardScaler().fit_transform(bc.data)
acum = np.cumsum(PCA().fit(Xbc).explained_variance_ratio_)
n95 = int(np.argmax(acum >= 0.95)) + 1
print("componentes para 95% da variancia:", n95, "de 30")
print("compressao de", round(30/n95, 1), "x perdendo so 5% da variacao.")

## Exercício 4 — PCA antes do t-SNE

In [ ]:
# @title Solução
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
import time

dig = load_digits()
rng = np.random.RandomState(SEMENTE)
sel = rng.choice(len(dig.data), 500, replace=False)
Xd = dig.data[sel]
t0 = time.time()
TSNE(n_components=2, init="random", random_state=SEMENTE).fit_transform(Xd)
direto = time.time() - t0
t0 = time.time()
X20 = PCA(n_components=20).fit_transform(Xd)
TSNE(n_components=2, init="random", random_state=SEMENTE).fit_transform(X20)
com_pca = time.time() - t0
print("t-SNE direto (64 dims):", round(direto, 2), "s")
print("PCA(20) + t-SNE:       ", round(com_pca, 2), "s")
print("distancias entre grupos no mapa NAO sao confiaveis (so vizinhanca local).")